# Optimización del calendario de la Premier League

Notebook general de uso para [modelo_calendario.py](../modelo_calendario.py). Selecciona una temporada y una URL CSV; el módulo descarga la base, obtiene los equipos y resuelve un nuevo calendario doble Round-Robin con Gurobi.

El CSV histórico se utiliza para identificar los equipos, no para fijar los partidos del calendario optimizado: Football-Data no proporciona una jornada oficial fiable en estos archivos. Las listas `clasicos`, `estadios_compartidos`, `equipos_europeos` y `jornadas_criticas` son datos de negocio que deben completarse cuando estén disponibles.

In [3]:
# Paso 1: localizar e importar el modelo general compartido.
from pathlib import Path
import sys

_directorios = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
RUTA_PROYECTO = next(
    (directorio for directorio in _directorios if (directorio / "modelo_calendario.py").is_file()),
    None,
)
if RUTA_PROYECTO is None:
    raise FileNotFoundError("No se encontró modelo_calendario.py en el proyecto ni sus carpetas padre.")
sys.path.insert(0, str(RUTA_PROYECTO))

from modelo_calendario import resolver_instancia

URLS_TEMPORADA = {
    "2021-2022": "https://www.football-data.co.uk/mmz4281/2122/E0.csv",
    "2022-2023": "https://www.football-data.co.uk/mmz4281/2223/E0.csv",
    "2023-2024": "https://www.football-data.co.uk/mmz4281/2324/E0.csv",
    "2024-2025": "https://www.football-data.co.uk/mmz4281/2425/E0.csv",
    "2025-2026": "https://www.football-data.co.uk/mmz4281/2526/E0.csv",
    "2026-2027": "https://www.football-data.co.uk/mmz4281/2627/E0.csv",
}
TEMPORADA = "2024-2025"
URL_CSV = URLS_TEMPORADA[TEMPORADA]
print(f"Instancia seleccionada: {TEMPORADA}")

Instancia seleccionada: 2024-2025


In [ ]:
# Paso 2: configurar restricciones operativas y pesos del objetivo.
# Completa estas listas si cuentas con la información específica del torneo.
PARAMETROS_MODELO = {
    "clasicos": [],  # Ejemplo: [("Arsenal", "Tottenham")]
    "estadios_compartidos": [],
    "equipos_europeos": [],
    "jornadas_criticas": [],  # Números de jornada entre 1 y 38
    "max_clasicos_por_jornada": 1,
    "peso_breaks": 1.0,
    "peso_viajes": 1.0,
    "tiempo_limite": 60,
    "numero_equipos_esperado": 20,
    "mostrar_log": True,
}

In [ ]:
# Paso 3: descargar la temporada seleccionada y resolver el modelo.
resultado = resolver_instancia(TEMPORADA, URL_CSV, **PARAMETROS_MODELO)

print(f"Estado Gurobi: {resultado['estado']}")
print(f"Partidos leídos desde internet: {len(resultado['partidos_historicos'])}")
print(f"Objetivo: {resultado['objetivo']}")
print(f"Breaks: {resultado['breaks']}")
print(f"Viajes críticos: {resultado['viajes_criticos']}")
display(resultado["calendario"])
display(resultado["partidos_historicos"][["Date", "HomeTeam", "AwayTeam"]].head(10))